## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [9]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


all dependencies present
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 3 — Feature Engineering

## This notebook no longer produces a modelling dataset

That is the single most important change in the whole remediation, so it gets
stated plainly.

GATE-1 failed because R01 Notebook 3 called `scaler.fit_transform(...)` and
`encoder.fit_transform(df[col])` on the whole frame and wrote
`engineered_data.csv`, which Notebook 6b then split. Every encoder, scaler,
median and composite had already seen the held-out rows, so every number in
Tables 3, 4 and 6 came from a leaked pipeline.

**You cannot fix that by patching this notebook.** The defect is the boundary
between this notebook and Notebook 6b: as long as a fitted artefact crosses
that boundary as a CSV, the leak exists. So feature engineering now lives in
`pipeline.preprocess_inside_fold()` and is called inside every fold by every
downstream notebook.

What this notebook does instead:

1. **Documents** the three composites with every governing constant printed,
   which is the Q3 fix (J11 breach — undisclosed constants).
2. **Validates** the pipeline on one fold and asserts the fold-safety
   properties that GATE-1 requires.
3. **Demonstrates** the leak, quantitatively, so M6 can state its size
   instead of asserting it was immaterial.
4. Writes an EDA-only file, clearly named, that no modelling notebook reads.

## The two-composite problem the examination did not catch

R01 Notebook 3 mapped income ordinally (`low:0, medium:1, high:2, hgh:2`).
R01 Notebook 9 — the *corrected* in-fold pipeline — label-encoded it
alphabetically instead. So `socioeconomic_vulnerability_score` had **two
different definitions**, and Table 3/4 and Table 5 were computed on different
features. That is a second, undiagnosed reason the two seed-42 rows disagree,
on top of the leakage fix. Both definitions now come from `config.ORDINAL_MAPS`.

In [10]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      binarise_target, raw_feature_cols)

banner("NOTEBOOK 3 — FEATURE ENGINEERING (fold-safe)")
OUT = run_dir("notebook03_features")
print("outputs ->", OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
print(f"train_pool {train_pool.shape}   test_holdout {test_holdout.shape}")

NOTEBOOK 3 — FEATURE ENGINEERING (fold-safe)
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
outputs -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/results/notebook03_features/20260921T134436Z_records
train_pool (784, 42)   test_holdout (197, 42)


In [12]:
# ---- 1. THE COMPOSITE SPECIFICATION (Q3 — print every constant) --------
spec = f"""
COMPOSITE FEATURE SPECIFICATION
================================================================

1. attendance_risk_index
   formula : weighted mean over terms of (1 - attendance_fraction)
   where   : attendance_fraction = clip(attendance, 0, {ATTENDANCE_MAX}) / {ATTENDANCE_MAX}
   terms   : {ATTENDANCE_COMPOSITE_COLS}   (TERM 3 EXCLUDED)
   weights : term_1 = {ATTENDANCE_WEIGHTS[0]}, term_2 = {ATTENDANCE_WEIGHTS[1]}
   why no term 3 : all 48 pupils with 0% term-3 attendance are dropouts. They
             had left before term 3 began, so term-3 attendance records the
             outcome rather than predicting it. R01 gave term 3 the LARGEST
             weight ({R01_ATTENDANCE_WEIGHTS[2]}) -- on the column most
             contaminated by the outcome.
   rationale : the more recent term carries more weight, because a late
             decline signals departure more strongly than an early dip that
             later recovers. That makes this feature an operationalisation of
             the claim that TRAJECTORY matters more than LEVEL -- which is
             Finn's central claim, not Bronfenbrenner's.
   R01 fault : attendance was divided by 100 only when max > 1.0, so values
             above 100% produced a NEGATIVE risk contribution. Now clipped first.

2. socioeconomic_vulnerability_score
   formula : (leap_flag + feeding_flag + income_vulnerability) / 3
   where   : income_vulnerability = 1 - (income_ordinal / {max(ORDINAL_MAPS[SOCIOECONOMIC_COLS['family_income']].values())})
   ordinal : {ORDINAL_MAPS[SOCIOECONOMIC_COLS['family_income']]}
   unknown : 'Unknown' -> NaN + a separate family_income_level_missing flag
   R01 fault : Notebook 9 divided a LABEL-ENCODED income column by its
             training max. Label encoding is alphabetical, so with both
             'High' and the misspelling 'Hgh' present the order was
             (Don't know, High, Hgh, Low, Medium) and the composite was not
             monotone in income. Notebook 3 used a different, ordinal map.
             Two definitions, two sets of results, one feature name.

3. behavioral_engagement_index
   formula : ((1 - warnings_scaled) + participation_scaled + extracurricular_scaled) / 3
   scaling : MinMaxScaler FITTED ON THE TRAINING FOLD ONLY, applied to the
             validation fold. This was the fit_transform-on-everything call
             that GATE-1 cited.
   free text : {STRING_TO_NUMERIC}

ALL THREE are built inside pipeline.preprocess_inside_fold(), step 11.
None is ever computed on the full dataset for modelling purposes.
"""
print(spec)
(OUT / "composite_specification.txt").write_text(spec)
print("Paste this into M7. The R01 M7 described the weights as 'giving greater "
      "weight to the most recent term' without printing them.")


COMPOSITE FEATURE SPECIFICATION

1. attendance_risk_index
   formula : weighted mean over terms of (1 - attendance_fraction)
   where   : attendance_fraction = clip(attendance, 0, 100.0) / 100.0
   terms   : ['term_1_attendance', 'term_2_attendance']   (TERM 3 EXCLUDED)
   weights : term_1 = 0.4, term_2 = 0.6
   why no term 3 : all 48 pupils with 0% term-3 attendance are dropouts. They
             had left before term 3 began, so term-3 attendance records the
             outcome rather than predicting it. R01 gave term 3 the LARGEST
             weight (0.45) -- on the column most
             contaminated by the outcome.
   rationale : the more recent term carries more weight, because a late
             decline signals departure more strongly than an early dip that
             later recovers. That makes this feature an operationalisation of
             the claim that TRAJECTORY matters more than LEVEL -- which is
             Finn's central claim, not Bronfenbrenner's.
   R01 fa

In [13]:
# ---- 2. VALIDATE fold safety (the GATE-1 assertions) -------------------
splits = cv_splits(train_pool, SPLIT_SEED)
tr, vl = splits[0]
X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl], verbose=True)

checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}  {detail}")

print("\nFOLD-SAFETY CHECKS")
check("validation columns identical to training, same order",
      list(X_vl.columns) == list(X_tr.columns), f"{X_tr.shape[1]} features")
check("no NaN in training fold", X_tr.isna().sum().sum() == 0)
check("no NaN in validation fold", X_vl.isna().sum().sum() == 0)
check(f"expected composites built for FEATURE_SET={FEATURE_SET!r}",
      set(meta["composites_present"]) == set(ACTIVE_COMPOSITES),
      str(meta["composites_present"]))
check("no questionnaire item in a records-only model",
      FEATURE_SET != "records" or not any(c in X_tr.columns for c in QUESTIONNAIRE_COLS))
check("no leakage column survived",
      not any(c in X_tr.columns for c in LEAKAGE_EXACT))
check("school_code handled per config",
      (SCHOOL_COL not in X_tr.columns) if SCHOOL_HANDLING == "drop" else True,
      f"SCHOOL_HANDLING={SCHOOL_HANDLING!r}")
check("attendance within physical bound",
      all(X_tr[c].max() <= ATTENDANCE_MAX for c in ATTENDANCE_COLS if c in X_tr))
check("no train/val row index overlap", len(set(tr) & set(vl)) == 0)

# the decisive one: does the validation fold influence any fitted statistic?
X_tr2, y_tr2, _, _, _ = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl].sample(frac=0.5, random_state=1))
check("training fold unchanged when the validation fold changes",
      X_tr.shape == X_tr2.shape and np.allclose(
          X_tr.select_dtypes(include=np.number).to_numpy(),
          X_tr2.select_dtypes(include=np.number).to_numpy(), equal_nan=True),
      "<- this is the property GATE-1 requires")

chk = pd.DataFrame(checks)
chk.to_csv(OUT / "fold_safety_checks.csv", index=False)
assert chk["pass"].all(), "fold-safety checks failed; fix before modelling"
print(f"\nall {len(chk)} checks pass")
print(f"features per fold: {meta['n_features']}  "
      f"(raw {len(raw_feature_cols(X_tr))} + composites {len(meta['composites_present'])})")
print("This is the number M2/M7 should state as d, computed per fold.")

    features=15 composites=1 clipped=3

FOLD-SAFETY CHECKS
  [PASS] validation columns identical to training, same order  15 features
  [PASS] no NaN in training fold  
  [PASS] no NaN in validation fold  
  [PASS] expected composites built for FEATURE_SET='records'  ['attendance_risk_index']
  [PASS] no questionnaire item in a records-only model  
  [PASS] no leakage column survived  
  [PASS] school_code handled per config  SCHOOL_HANDLING='drop'
  [PASS] attendance within physical bound  
  [PASS] no train/val row index overlap  
  [PASS] training fold unchanged when the validation fold changes  <- this is the property GATE-1 requires

all 10 checks pass
features per fold: 15  (raw 14 + composites 1)
This is the number M2/M7 should state as d, computed per fold.


In [14]:
# ---- 3. QUANTIFY THE R01 LEAK (so M6 can state its size) ---------------
# M6 asserted the corrected pipeline left the comparison "materially
# unchanged". Table 5 contradicted that. Measure it instead of asserting it.
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

def preprocess_leaky(df_all, tr_idx, vl_idx):
    """The R01 arrangement: fit everything on the FULL frame, then split."""
    d = df_all.copy()
    d[TARGET] = binarise_target(d[TARGET])
    # R01 kept term 3, so the reproduction must too
    d = d.drop(columns=[c for c in d.columns if c != TARGET and drop_reason(c)
                        and not drop_reason(c).startswith("temporal")],
               errors="ignore")
    if SCHOOL_HANDLING == "drop" and SCHOOL_COL in d.columns:
        d = d.drop(columns=[SCHOOL_COL])
    y = d[TARGET]; X = d.drop(columns=[TARGET])
    num = X.select_dtypes(include=np.number).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    X[num] = X[num].fillna(X[num].median())                 # full-frame median
    for c in cat:
        m = X[c].mode()
        X[c] = X[c].fillna(m.iloc[0] if len(m) else "unknown")
        X[c] = LabelEncoder().fit_transform(X[c].astype(str))  # full-frame fit
    att = [c for c in R01_ATTENDANCE_COLS if c in X.columns]
    if len(att) == 3:
        m = X[att].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
        if np.nanmax(m) > 1.0:
            m = m / 100.0                      # R01: no clipping, so >100 -> negative
        X["attendance_risk_index"] = np.average(1.0 - m, axis=1,
                                                weights=R01_ATTENDANCE_WEIGHTS)
    inc = SOCIOECONOMIC_COLS["family_income"]
    leap = SOCIOECONOMIC_COLS["leap_beneficiary"]
    feed = SOCIOECONOMIC_COLS["school_feeding"]
    if all(c in X.columns for c in (inc, leap, feed)):
        vuln = 1.0 - (X[inc].astype(float) / X[inc].max())   # alphabetical ints
        X["socioeconomic_vulnerability_score"] = (
            X[leap].astype(float) + X[feed].astype(float) + vuln) / 3.0
    b = [BEHAVIOR_COLS[k] for k in ("behavior_warnings", "class_participation",
                                     "extracurricular")]
    if all(c in X.columns for c in b):
        s = MinMaxScaler().fit_transform(                    # full-frame fit
            X[b].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float))
        X["behavioral_engagement_index"] = (
            (1 - s[:, 0]) + s[:, 1] + s[:, 2]) / 3.0
    return (X.iloc[tr_idx], y.iloc[tr_idx].astype(int),
            X.iloc[vl_idx], y.iloc[vl_idx].astype(int))

Xl_tr, yl_tr, Xl_vl, yl_vl = preprocess_leaky(train_pool, tr, vl)

rows = []
for name, a, b in [
    ("attendance_risk_index", X_vl.get("attendance_risk_index"),
     Xl_vl.get("attendance_risk_index")),
    ("socioeconomic_vulnerability_score", X_vl.get("socioeconomic_vulnerability_score"),
     Xl_vl.get("socioeconomic_vulnerability_score")),
    ("behavioral_engagement_index", X_vl.get("behavioral_engagement_index"),
     Xl_vl.get("behavioral_engagement_index")),
]:
    if a is None or b is None:
        continue
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    rows.append({"composite": name,
                 "in_fold_mean": a.mean(), "leaky_mean": b.mean(),
                 "mean_abs_difference": np.abs(a - b).mean(),
                 "pearson_r": float(np.corrcoef(a, b)[0, 1]),
                 "identical": bool(np.allclose(a, b))})
leak_df = pd.DataFrame(rows)
leak_df.to_csv(OUT / "leak_magnitude_composites.csv", index=False)
print("Composite values: corrected in-fold vs the R01 leaky arrangement")
print(leak_df.round(4).to_string(index=False))
print("\nattendance_risk_index differs because R01 built it from three terms "
      "including term 3; the corrected version uses terms 1-2 only. A low "
      "pearson_r on socioeconomic_vulnerability_score is the income-encoding "
      "fault. These are separate defects R01 confounded — report each in M6.")

Composite values: corrected in-fold vs the R01 leaky arrangement
            composite  in_fold_mean  leaky_mean  mean_abs_difference  pearson_r  identical
attendance_risk_index        0.1778      0.1861               0.0229     0.9849      False

attendance_risk_index differs because R01 built it from three terms including term 3; the corrected version uses terms 1-2 only. A low pearson_r on socioeconomic_vulnerability_score is the income-encoding fault. These are separate defects R01 confounded — report each in M6.


In [15]:
# ---- 4. EDA-only export (read by no modelling notebook) ----------------
X_eda, y_eda, _, _, meta_eda = preprocess_inside_fold(
    train_pool, test_holdout.head(1))     # fitted on the training pool only
eda_out = X_eda.copy(); eda_out[TARGET] = y_eda.to_numpy()
eda_out.to_csv(ENGINEERED_EDA_CSV, index=False)

print(f"wrote {ENGINEERED_EDA_CSV.name}  {eda_out.shape}")
print("\n" + "!"*72)
print("THIS FILE IS FOR FIGURES AND DESCRIPTIVE TABLES ONLY.")
print("No modelling notebook reads it. Feeding it to a model and then")
print("splitting is exactly the R01 arrangement that failed GATE-1.")
print("!"*72)

print("\ncomposite descriptives (training pool, for Table 2):")
print(eda_out[[c for c in COMPOSITES if c in eda_out]]
      .describe().T[["mean", "std", "min", "max"]].round(3).to_string())
print("\ncorrelation with target:")
print(eda_out[[c for c in COMPOSITES if c in eda_out] + [TARGET]]
      .corr()[TARGET].drop(TARGET).round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, c in zip(axes, [c for c in COMPOSITES if c in eda_out]):
    sns.kdeplot(data=eda_out, x=c, hue=TARGET, common_norm=False, ax=ax)
    ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "figures/composite_distributions.png", dpi=200)
plt.close()

write_manifest(OUT, {
    "notebook": "03_features",
    "produces_modelling_dataset": False,
    "n_features_per_fold": int(meta["n_features"]),
    "composites": meta["composites_present"],
    "fold_safety_all_pass": bool(chk["pass"].all()),
    "attendance_weights": ATTENDANCE_WEIGHTS.tolist(),
    "income_ordinal_map": ORDINAL_MAPS.get(SOCIOECONOMIC_COLS["family_income"]),
})
print("\nNEXT: Notebook 4 (baselines, in-fold, training pool only).")

wrote engineered_data_FOR_EDA_ONLY.csv  (784, 16)

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
THIS FILE IS FOR FIGURES AND DESCRIPTIVE TABLES ONLY.
No modelling notebook reads it. Feeding it to a model and then
splitting is exactly the R01 arrangement that failed GATE-1.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

composite descriptives (training pool, for Table 2):
                        mean   std  min   max
attendance_risk_index  0.174  0.14  0.0  0.86

correlation with target:
attendance_risk_index    0.743

NEXT: Notebook 4 (baselines, in-fold, training pool only).


---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [16]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook03_features/20260921T134436Z_records
files written : 5
    RUN_MANIFEST.json
    composite_specification.txt
    figures/composite_distributions.png
    fold_safety_checks.csv
    leak_magnitude_composites.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all